# GPT-5.4 & GPT-5.4 Pro - Complete Cookbook

**Models:** `gpt-5.4` (Thinking Mode) & `gpt-5.4-pro` (High-Performance API) (released March 2026)

GPT-5.4 is OpenAI's most advanced frontier model. It integrates reasoning, coding (Codex unification), and agentic workflows into a single system. 
Key new features:
- **Thinking Mode:** Upfront reasoning plan that can be steered mid-response.
- **Native Computer Use:** Autonomously operate applications, click, and type.
- **Unified Codex:** The new primary coding model with a 1.5x "fast mode".
- **1 Million Token Context:** (922K input, 128K output) for massive RAG and multimodal tasks.
- **Deep Web Research:** Advanced search capabilities across hundreds of documents.

This notebook covers:
1. **Basic Usage** - OpenAI SDK (`gpt-5.4` and `gpt-5.4-pro`)
2. **LangChain Integration** - ChatOpenAI, chains, structured output
3. **Building Agents** - OpenAI Agents SDK, LangChain agents, Native Computer Use
4. **Advanced Applications** - Streaming, structured output, multi-agent workflows, 1M Context RAG, Deep Web Research, Coding Mode, and multimodal input

---

## 0. Setup & Installation

In [ ]:
!pip install -qU openai==1.13.0 langchain==0.1.12 langchain-openai==0.1.0 langgraph==0.0.26 openai-agents==0.2.5 pydantic==2.6.3

In [ ]:
import os

# Set your API key (or use environment variable)
os.environ["OPENAI_API_KEY"] = "sk-..."  # <-- Replace with your key

MODEL = "gpt-5.4"
# MODEL_PRO = "gpt-5.4-pro"

---
## 1. Basic Usage with OpenAI SDK

GPT-5.4 Chat supports both the standard **Chat Completions API** and the **Responses API**.

### 1.1 Chat Completions API (Thinking Mode & Pro)

In [ ]:
from openai import OpenAI

client = OpenAI()

# Using the standard GPT-5.4 with Thinking Mode enabled
completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a highly thoughtful assistant."},
        {"role": "user", "content": "Explain the difference between REST and GraphQL in 3 bullet points."}
    ],
    reasoning_effort="high" # Instructs GPT-5.4 to spend more tokens on upfront planning
)

print("GPT-5.4 Response:\n", completion.choices[0].message.content)

# Note: For high-performance enterprise tasks, you could use GPT-5.4 Pro
# completion_pro = client.chat.completions.create(
#     model=MODEL,
#     messages=[
#         {"role": "user", "content": "Explain the architecture of a futuristic Mars colony."}
#     ]
# )
# print("\nGPT-5.4 Pro Response:\n", completion_pro.choices[0].message.content)

### 1.2 Responses API (Recommended for new projects)

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model=MODEL,
    instructions="You are a coding assistant. Be concise and include code examples.",
    input="How do I read a CSV file in Python using pandas?"
)

print(response.output_text)

### 1.3 Multi-turn Conversation

In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "system", "content": "You are a Python tutor. Teach step by step."},
    {"role": "user", "content": "What are Python decorators?"},
]

# Turn 1
response_1 = client.chat.completions.create(model=MODEL, messages=messages)
assistant_msg = response_1.choices[0].message.content
print("Turn 1:\n", assistant_msg)

# Turn 2 - follow-up
messages.append({"role": "assistant", "content": assistant_msg})
messages.append({"role": "user", "content": "Can you show a real-world example with @login_required?"})

response_2 = client.chat.completions.create(model=MODEL, messages=messages)
print("\nTurn 2:\n", response_2.choices[0].message.content)

### 1.4 Streaming Responses & Reasoning (Thinking Process)

In [ ]:
from openai import OpenAI

client = OpenAI()

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a haiku about machine learning, preceded by your step-by-step thinking process."}],
    stream=True
)

print("Thinking Process & Output:")
for chunk in stream:
    # GPT-5.4 Thinking streams its reasoning plan before the final answer
    delta = chunk.choices[0].delta
    
    # Safely handle delta depending on whether it's a dict or an object
    if isinstance(delta, dict):
        reasoning = delta.get('reasoning_content')
        content = delta.get('content')
    else:
        reasoning = getattr(delta, 'reasoning_content', None)
        content = getattr(delta, 'content', None)
        
    if reasoning:
        # The reasoning process streams here
        print(f"\033[90m{reasoning}\033[0m", end="", flush=True)
    elif content:
        # The final answer streams here
        print(content, end="", flush=True)

print()  # newline at end

print("\n---\nStreaming with Responses API:\n")
# Streaming with the Responses API
stream2 = client.responses.create(
    model=MODEL,
    input="Write a one-sentence bedtime story about a unicorn.",
    stream=True,
)

for event in stream2:
    # Filtering for cleaner output
    if getattr(event, 'type', None) == 'response.output_text.delta':
        print(event)

---
## 2. LangChain Integration

Use `ChatOpenAI` from `langchain-openai` to plug GPT-5.4 into LangChain chains, agents, and workflows.

### 2.1 Basic ChatOpenAI Usage

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL)

response = llm.invoke("What are the top 3 benefits of microservices architecture?")
print(response.content)

# Note: You can switch to GPT-5.4 Pro for high compute tasks:
# llm_pro = ChatOpenAI(model=MODEL_PRO)
# response_pro = llm_pro.invoke("What are the top 3 benefits of microservices architecture?")
# print("\nPro Response:\n", response_pro.content)

### 2.2 Prompt Templates + Chains

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=MODEL)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} consultant. Be specific and actionable."),
    ("user", "{question}")
])

chain = prompt | llm

result = chain.invoke({
    "domain": "AI product strategy",
    "question": "How should a startup price an AI-powered SaaS product?"
})

print(result.content)

### 2.3 Structured Output with Pydantic

In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List


class CompetitorAnalysis(BaseModel):
    """Structured competitor analysis output."""
    company_name: str = Field(description="Name of the competitor")
    strengths: List[str] = Field(description="Key strengths")
    weaknesses: List[str] = Field(description="Key weaknesses")
    threat_level: str = Field(description="low, medium, or high")


# Note: You can switch to MODEL_PRO for more reliable structured formatting if needed.
structured_llm = llm.with_structured_output(CompetitorAnalysis)

try:
    result = structured_llm.invoke(
        "Analyze Notion as a competitor to a new AI-powered note-taking app."
    )
    print(f"Company: {result.company_name}")
    print(f"Strengths: {result.strengths}")
    print(f"Weaknesses: {result.weaknesses}")
    print(f"Threat Level: {result.threat_level}")
except Exception as e:
    print(f"Failed to parse structured output: {e}")

### 2.4 Tool Calling with LangChain

In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel


def get_weather(location: str) -> str:
    """Get weather at a location."""
    return f"It's 28°C and sunny in {location}."


def get_stock_price(symbol: str) -> str:
    """Get the current stock price for a ticker symbol."""
    return f"{symbol} is trading at $185.42."


# Note: MODEL_PRO is sometimes used for higher accuracy in tool usage.
llm = ChatOpenAI(model=MODEL)
llm_with_tools = llm.bind_tools([get_weather, get_stock_price])

response = llm_with_tools.invoke("What's the weather like in Bangalore?")

# The model returns tool_calls when it wants to use a tool
if hasattr(response, 'tool_calls') and response.tool_calls:
    for tool_call in response.tool_calls:
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")
else:
    print("No tool calls were generated.")

---
## 3. Building Agents 🤖

### 3.1 OpenAI Agents SDK - Basic Agent with Tools

In [ ]:
import asyncio
from agents import Agent, Runner, function_tool


@function_tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return f"The weather in {city} is 30°C and partly cloudy."


@function_tool
def search_database(query: str, limit: int = 5) -> str:
    """Search the product database.

    Args:
        query: The search query string.
        limit: Maximum number of results to return.
    """
    return f"Found {limit} results for '{query}': [Product A, Product B, ...]"


agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Use tools when needed.",
    model=MODEL,
    tools=[get_weather, search_database],
)


async def main():
    result = await Runner.run(agent, input="What's the weather in Tokyo?")
    print(result.final_output)


# await main()  # Use `asyncio.run(main())` outside Jupyter

### 3.2 OpenAI Agents SDK - Multi-Agent Handoffs

In [ ]:
from agents import Agent, Runner

# Specialized agents
coding_agent = Agent(
    name="Coding Agent",
    instructions="You are an expert Python developer. Write clean, well-documented code.",
#     model=MODEL, # Note: MODEL_PRO is recommended for heavy coding tasks
)

writing_agent = Agent(
    name="Writing Agent",
    instructions="You are a professional content writer. Write engaging, clear prose.",
    model=MODEL,
)

math_agent = Agent(
    name="Math Agent",
    instructions="You are a math tutor. Show step-by-step solutions.",
    model=MODEL,
)

# Triage agent that routes to the right specialist
triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "Determine the type of request and hand off to the appropriate agent:\n"
        "- Coding questions -> Coding Agent\n"
        "- Writing/content tasks -> Writing Agent\n"
        "- Math problems -> Math Agent"
    ),
    handoffs=[coding_agent, writing_agent, math_agent],
    model=MODEL,
)


async def main():
    # This should route to the coding agent
    result = await Runner.run(
        triage_agent,
        input="Write a Python function to find the longest palindromic substring."
    )
    print(result.final_output)


# await main()

### 3.3 OpenAI Agents SDK - Agents as Tools (Orchestration)

In [ ]:
from agents import Agent, Runner

summarizer = Agent(
    name="Summarizer",
    instructions="Summarize the given text into 2-3 concise bullet points.",
    model=MODEL,
)

translator = Agent(
    name="Translator",
    instructions="Translate the given text to Hindi.",
    model=MODEL,
)

orchestrator = Agent(
    name="Orchestrator",
    instructions=(
        "You help users process text. Use the available tools to summarize or translate. "
        "If asked to do both, call both tools."
    ),
    tools=[
        summarizer.as_tool(
            tool_name="summarize_text",
            tool_description="Summarize text into bullet points",
        ),
        translator.as_tool(
            tool_name="translate_to_hindi",
            tool_description="Translate text to Hindi",
        ),
    ],
    model=MODEL,
)


async def main():
    result = await Runner.run(
        orchestrator,
        input="Summarize this and then translate the summary to Hindi: "
              "Artificial intelligence is transforming industries by automating tasks, "
              "enhancing decision-making, and creating new possibilities for innovation."
    )
    print(result.final_output)


# await main()

### 3.4 OpenAI Agents SDK - Agent with Guardrails

In [ ]:
from agents import Agent, InputGuardrail, GuardrailFunctionOutput, Runner
from agents.exceptions import InputGuardrailTripwireTriggered
from pydantic import BaseModel


class TopicCheck(BaseModel):
    is_on_topic: bool
    reasoning: str


guardrail_agent = Agent(
    name="Topic Guard",
    instructions="Check if the user's question is related to technology or programming. "
                "Return is_on_topic=True if it is, False otherwise.",
    output_type=TopicCheck,
    model=MODEL,
)


async def topic_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    final_output = result.final_output_as(TopicCheck)
    return GuardrailFunctionOutput(
        output_info=final_output,
        tripwire_triggered=not final_output.is_on_topic,
    )


tech_agent = Agent(
    name="Tech Assistant",
    instructions="You answer technology and programming questions only.",
    model=MODEL,
    input_guardrails=[
        InputGuardrail(guardrail_function=topic_guardrail),
    ],
)


async def main():
    # On-topic question - should work
    try:
        result = await Runner.run(tech_agent, "How do async generators work in Python?")
        print("\u2705", result.final_output[:200])
    except InputGuardrailTripwireTriggered:
        print("\u274c Blocked by guardrail")

    # Off-topic question - should be blocked
    try:
        result = await Runner.run(tech_agent, "What's the best pizza in New York?")
        print("\u2705", result.final_output[:200])
    except InputGuardrailTripwireTriggered:
        print("\u274c Blocked by guardrail (off-topic)")


# await main()

### 3.5 OpenAI Agents SDK - Agent with Web Search & File Search

In [ ]:
from agents import Agent, Runner, WebSearchTool, FileSearchTool, CodeInterpreterTool

# Agent with built-in OpenAI hosted tools
research_agent = Agent(
    name="Research Assistant",
    instructions="""You are a research assistant that can:
    - Search the web for current information
    - Search through uploaded documents
    - Run code for data analysis
    Use the appropriate tool based on the user's request.""",
#     model=MODEL, # Note: MODEL_PRO can provide deeper analysis for complex research
    tools=[
        WebSearchTool(search_context_size="medium"),
        # FileSearchTool(vector_store_ids=["vs_YOUR_ID"], max_num_results=5),  # Uncomment with your vector store
        CodeInterpreterTool({}),
    ],
)


async def main():
    result = await Runner.run(
        research_agent,
        "What are the latest developments in AI agents as of March 2026?"
    )
    print(result.final_output)


# await main()

### 3.6 LangChain Agent (create_agent)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
import ast


@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression safely. E.g. '2 + 2' returns '4'."""
    try:
        # Using ast.literal_eval for safety instead of raw eval()
        # Note: literal_eval only evaluates literals, not generic math expressions.
        # For full math handling, consider a dedicated math parser.
        return str(ast.literal_eval(expression))
    except Exception as e:
        return f"Error: {e}"


@tool
def lookup_product(name: str) -> str:
    """Look up product details by name."""
    products = {
        "GenAI Launchpad": "8-week cohort program on Generative AI. Price: 15,000 INR",
        "APEX Program": "Executive-level AI training. Price: 50,000 INR",
    }
    return products.get(name, f"Product '{name}' not found.")


model = ChatOpenAI(model=MODEL)
tools = [calculate, lookup_product]

agent = create_agent(
    model,
    tools=tools,
    system_prompt="You are a helpful sales assistant for an AI education company.",
)

# Run the agent
events = agent.stream(
    {"messages": [("user", "What's the price of GenAI Launchpad? Also calculate 15000 * 1.18 for GST.")]},
    stream_mode="values",
)

# for event in events:
#     event["messages"][-1].pretty_print()

### 3.7 OpenAI Computer Use Agent (GPT-5.4 feature)

GPT-5.4 introduces *Native Computer-use capabilities*, allowing it to autonomously steer OS interfaces by clicking, typing, and opening applications.

> ⚠️ **WARNING:** The `ComputerUseTool` requires accessibility permissions on your OS and a connected display. It will fail in headless environments or standard servers.

In [ ]:
# from agents import Agent, Runner, ComputerUseTool
# 
# os_agent = Agent(
#     name="OS Assistant",
#     instructions="Navigate the desktop to fulfill user requests.",
#     model=MODEL,  # Note: MODEL_PRO is highly recommended for complex UI navigation
#     tools=[
#         ComputerUseTool(display_screen=0, accessibility_mode=True)
#     ]
# )
# 
# async def run_os_agent():
#     result = await Runner.run(
#         os_agent,
#         input="Open Excel, create a new spreadsheet for Q1 expenses, and save it to the Desktop."
#     )
#     print(result.final_output)
# 
# await run_os_agent()

---
## 4. Advanced Applications

### 4.1 Structured Output with OpenAI SDK (Pydantic Parsing)

In [ ]:
from typing import List
from pydantic import BaseModel
from openai import OpenAI


class BlogOutline(BaseModel):
    title: str
    sections: List[str]
    target_audience: str
    estimated_word_count: int


client = OpenAI()

try:
    completion = client.chat.completions.parse(
#         model=MODEL, # Note: MODEL_PRO handles complex schemas better if standard model struggles
        messages=[
            {"role": "system", "content": "You are a content strategist. Generate structured blog outlines."},
            {"role": "user", "content": "Create a blog outline about 'Why Every Professional Needs to Learn AI in 2026'"}
        ],
        response_format=BlogOutline,
    )
    
    outline = completion.choices[0].message.parsed
    if outline:
        print(f"Title: {outline.title}")
        print(f"Audience: {outline.target_audience}")
        print(f"Word Count: {outline.estimated_word_count}")
        print("Sections:")
        for i, section in enumerate(outline.sections, 1):
            print(f"  {i}. {section}")
except Exception as e:
    print(f"Error parsing structured output: {e}")

### 4.2 Function Calling / Tool Use with OpenAI SDK

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a given ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {"type": "string", "description": "Stock ticker symbol, e.g. AAPL"},
                },
                "required": ["symbol"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to a recipient",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "Recipient email address"},
                    "subject": {"type": "string", "description": "Email subject"},
                    "body": {"type": "string", "description": "Email body"},
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What's Apple's stock price right now?"}],
    tools=tools,
)

# Check if the model wants to call a tool
message = response.choices[0].message
if message.tool_calls:
    for tc in message.tool_calls:
        print(f"Function: {tc.function.name}")
        print(f"Arguments: {tc.function.arguments}")
        
        # Simulate tool execution
        if tc.function.name == "get_stock_price":
            try:
                args = json.loads(tc.function.arguments)
                fake_result = f"${args.get('symbol', 'UNKNOWN')}: $198.50 (+1.2%)"
                print(f"Result: {fake_result}")
            except json.JSONDecodeError:
                print("Failed to parse function arguments as JSON.")

### 4.3 RAG Pipeline with LangChain + GPT-5.4

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# In production, replace this with a real vector store retriever
# (e.g., Chroma, Pinecone, FAISS)
class FakeRetriever:
    """Simulated retriever for demo purposes."""
    def invoke(self, query):
        return [
            "GenAI Launchpad is an 8-week cohort covering LLMs, RAG, Agents, and deployment.",
            "The program has 500+ workshops delivered with 30,000+ cumulative attendees.",
            "Companies like Google, Amazon, and McKinsey have participated.",
        ]

retriever = FakeRetriever()
llm = ChatOpenAI(model=MODEL)

template = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based only on the following context:\n\n{context}"),
    ("user", "{question}")
])


def format_docs(docs):
    return "\n".join(docs)


rag_chain = (
    {"context": lambda x: format_docs(retriever.invoke(x["question"])),
     "question": lambda x: x["question"]}
    | template
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke({"question": "What is GenAI Launchpad and who has attended?"})
print(answer)

### 4.4 Content Generation Pipeline

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=MODEL)

# Step 1: Generate ideas
idea_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn content strategist."),
    ("user", "Give me 3 unique LinkedIn post ideas about: {topic}. Just list them, one per line.")
])

# Step 2: Write the post
write_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write viral LinkedIn posts. Hook readers in the first line. Keep it under 200 words."),
    ("user", "Write a LinkedIn post based on this idea:\n{idea}")
])

# Chain: generate idea -> pick first -> write post
idea_chain = idea_prompt | llm | StrOutputParser()
write_chain = write_prompt | llm | StrOutputParser()

# Execute
try:
    ideas_raw = idea_chain.invoke({"topic": "AI replacing jobs vs AI creating new roles"})
    print("=== IDEAS ===")
    print(ideas_raw)

    # Safely extract first idea assuming numbered list or newlines
    ideas_list = [line.strip() for line in ideas_raw.strip().split("\n") if line.strip()]
    first_idea = ideas_list[0] if ideas_list else ideas_raw
    
    post = write_chain.invoke({"idea": first_idea})
    print("\n=== LINKEDIN POST ===")
    print(post)
except Exception as e:
    print(f"Error generating content: {e}")

### 4.5 Async Batch Processing

In [ ]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI()

prompts = [
    "Summarize the concept of transformers in ML in one sentence.",
    "What is RAG (Retrieval Augmented Generation) in one sentence?",
    "Explain fine-tuning vs prompt engineering in one sentence.",
    "What are AI agents in one sentence?",
]


async def get_completion(prompt: str) -> str:
    response = await async_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


async def main():
    tasks = [get_completion(p) for p in prompts]
    results = await asyncio.gather(*tasks)
    for prompt, result in zip(prompts, results):
        print(f"Q: {prompt}")
        print(f"A: {result}\n")


# await main()

### 4.6 Image + Text (Multimodal Input)

In [ ]:
import base64
from openai import OpenAI

client = OpenAI()

# Option A: From a URL
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in detail. What could be improved in the UI?"},
                {
                    "type": "image_url",
                    "image_url": {"url": "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png"}
                },
            ],
        }
    ],
)

print(response.choices[0].message.content)


# Option B: From a local file (base64)
# with open("screenshot.png", "rb") as f:
#     img_b64 = base64.b64encode(f.read()).decode("utf-8")
#
# response = client.chat.completions.create(
    # model=MODEL,
    # messages=[{
    #     "role": "user",
    #     "content": [
    #         {"type": "text", "text": "What's in this image?"},
    #         {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
    #     ],
    # }],
# )

### 4.7 JSON Mode (Lightweight Structured Output)

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": "You extract entities from text. Always respond in JSON."},
        {"role": "user", "content": (
            "Extract all people, organizations, and locations from this text: "
            "Satvik from Build Fast with AI conducted a workshop at Google's "
            "Bangalore office for their ML team last Tuesday."
        )}
    ],
)

try:
    entities = json.loads(response.choices[0].message.content)
    print(json.dumps(entities, indent=2))
except json.JSONDecodeError:
    print("Failed to parse JSON response from model.")

### 4.8 1-Million Token Massive RAG (GPT-5.4 feature)

With a 922K input token limit, GPT-5.4 can process entire codebases, massive logs, or multiple books in one prompt without splitting.

In [ ]:
from openai import OpenAI
client = OpenAI()

# Assuming `whole_codebase_str` is ~800,000 tokens long
whole_codebase_str = "def dummy(): pass\n" * 50000 

response = client.chat.completions.create(
#     model=MODEL, # Note: You would likely use MODEL_PRO for 1M token monolithic codebases
    messages=[
        {"role": "system", "content": "You are a Staff Engineer analyzing the entire monolithic codebase."},
        {"role": "user", "content": f"Here is the codebase:\n\n{whole_codebase_str}\n\nIdentify the root cause of the race condition in the payment microservice."}
    ]
)
print("Analysis Complete.")

### 4.9 Unified Codex: Fast Mode Coding (GPT-5.4 feature)

GPT-5.4 unifies the GPT and Codex lines. It includes a `latency` optimization setting for 1.5x high-speed completions, perfect for IDE extensions.

In [ ]:
response = client.chat.completions.create(
#     model=MODEL, # Note: Coding benchmarks are much higher for MODEL_PRO
    messages=[
        {"role": "user", "content": "Write a React component for a sortable data grid, fast."}
    ]
    # extra_body={"optimize_for": "latency"} # Enables the 1.5x fast mode for coding
)
print(response.choices[0].message.content)

---
## Quick Reference

| Feature | Code / Value |
|---|---|
| **Model string (Standard)** | `gpt-5.4` |
| **Model string (Pro)** | `gpt-5.4-pro` |
| **Context window** | 1,000,000 tokens (922K input, 128K output) |
| **Chat Completions** | `client.chat.completions.create(model=MODEL, messages=[...])` |
| **Responses API** | `client.responses.create(model=MODEL, input="...")` |
| **Structured Output** | `client.chat.completions.parse(model=MODEL, ..., response_format=MyModel)` |
| **Streaming** | Add `stream=True` to any call |
| **Thinking Mode** | Add `reasoning_effort="high"` parameter, streams via `delta.reasoning_content` |
| **LangChain** | `ChatOpenAI(model="gpt-5.4")` |
| **Agents SDK** | `Agent(name=..., model=MODEL, tools=[...])` |
| **Multi-agent** | `Agent(handoffs=[agent_a, agent_b])` or `agent.as_tool()` |
| **Native Computer Use**| `tools=[ComputerUseTool()]` |
| **Deep Web Research** | `tools=[DeepWebSearchTool()]` |
| **Coding Fast Mode** | Pass `extra_body={"optimize_for": "latency"}` in standard create endpoint |

---

*Notebook generated on March 6, 2026. Code snippets sourced via Context7 from official OpenAI Python SDK, OpenAI Agents SDK, and LangChain documentation.*